# 01. Pipeline Configuration & Multi-Environment Setup

Centralizes all architectural configurations with multi-environment isolation (Dev $\rightarrow$ Prod):
- **Target Environment**: Parameterized via `target_env` (`dev` | `prod` | `qa`), defaulting to **`dev`** for safe development.
- **Dynamic Catalogs**: `bronze_{target_env}`, `silver_{target_env}`, `gold_{target_env}` (e.g. `bronze_dev`, `silver_dev`, `gold_dev`).
- **Isolated Landing & Checkpoints**: `/Volumes/bronze_{env}/ecommerce/raw_landing/` and checkpoints.
- **Domain Schemas**: `customers`, `orders`, `products`, `region` in Silver; `sales`, `customers`, `products`, `reference` in Gold.
- **Zero-Copy Clone Utility**: `clone_prod_to_dev()` to snapshot production tables into dev in seconds.
- **Audit & Hashing Utilities**: SHA-256 change detection and ETL audit stamps.

Included in all downstream notebooks via `%run ./01_Config`


In [0]:
# Environment & Catalog Configuration Widgets
dbutils.widgets.dropdown("target_env", "dev", ["dev", "prod", "qa"], "Target Environment (dev | prod | qa)")
target_env = dbutils.widgets.get("target_env").lower().strip()

# Catalogs dynamically default to bronze_{env}, silver_{env}, gold_{env}
dbutils.widgets.text("bronze_catalog", f"bronze_{target_env}", "Bronze Catalog")
dbutils.widgets.text("silver_catalog", f"silver_{target_env}", "Silver Catalog")
dbutils.widgets.text("gold_catalog", f"gold_{target_env}", "Gold Catalog")
dbutils.widgets.text("source_schema", "ecommerce", "Source Schema")
dbutils.widgets.text("volume_name", "raw_landing", "Landing Volume")

bronze_catalog = dbutils.widgets.get("bronze_catalog")
silver_catalog = dbutils.widgets.get("silver_catalog")
gold_catalog = dbutils.widgets.get("gold_catalog")
source_schema = dbutils.widgets.get("source_schema")
volume_name = dbutils.widgets.get("volume_name")


In [0]:
# Constants & Paths
TARGET_ENV = target_env
BRONZE_CATALOG = bronze_catalog
SILVER_CATALOG = silver_catalog
GOLD_CATALOG = gold_catalog

BRONZE_SCHEMA = source_schema

# Silver Domain Schemas
SILVER_SCHEMAS = {
    "customers": "customers",
    "orders": "orders",
    "products": "products",
    "region": "region"
}

# Gold Consumption Schemas
GOLD_SCHEMAS = {
    "sales": "sales",
    "customers": "customers",
    "products": "products",
    "reference": "reference"
}

# Unity Catalog Volume Storage Locations
RAW_LANDING_VOLUME = f"/Volumes/{BRONZE_CATALOG}/{BRONZE_SCHEMA}/{volume_name}"
CHECKPOINT_VOLUME = f"/Volumes/{BRONZE_CATALOG}/{BRONZE_SCHEMA}/checkpoints"
API_LANDING_VOLUME = f"{RAW_LANDING_VOLUME}/api_orders"

# Spark Optimization Configurations
spark.conf.set("spark.databricks.delta.optimizeWrite.enabled", "true")
spark.conf.set("spark.databricks.delta.autoCompact.enabled", "true")

print(f"✓ Target Environment: {TARGET_ENV.upper()}")
print(f"✓ Configured Bronze:  {BRONZE_CATALOG}.{BRONZE_SCHEMA}")
print(f"✓ Configured Silver:  {SILVER_CATALOG}.{{{', '.join(SILVER_SCHEMAS.values())}}}")
print(f"✓ Configured Gold:    {GOLD_CATALOG}.{{{', '.join(GOLD_SCHEMAS.values())}}}")
print(f"✓ Landing Volume:    {RAW_LANDING_VOLUME}")
print(f"✓ Checkpoint Volume: {CHECKPOINT_VOLUME}")


In [0]:
def init_environment():
    """Initializes Unity Catalog catalogs, schemas, and volumes for the active environment if they do not exist."""
    print(f"[INIT] Initializing Unity Catalog environment for: {TARGET_ENV.upper()}...")
    
    # 1. Create Catalogs
    for cat in [BRONZE_CATALOG, SILVER_CATALOG, GOLD_CATALOG]:
        spark.sql(f"CREATE CATALOG IF NOT EXISTS {cat}")
        print(f"  ✓ Catalog: {cat}")
        
    # 2. Create Bronze Source Schema & Volumes
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {BRONZE_CATALOG}.{BRONZE_SCHEMA} COMMENT 'Raw source data for {BRONZE_SCHEMA} ({TARGET_ENV})'")
    spark.sql(f"CREATE VOLUME IF NOT EXISTS {BRONZE_CATALOG}.{BRONZE_SCHEMA}.{volume_name}")
    spark.sql(f"CREATE VOLUME IF NOT EXISTS {BRONZE_CATALOG}.{BRONZE_SCHEMA}.checkpoints")
    print(f"  ✓ Bronze Schema & Volumes: {BRONZE_CATALOG}.{BRONZE_SCHEMA}")
    
    # 3. Create Silver Domain Schemas
    for schema_name in SILVER_SCHEMAS.values():
        spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SILVER_CATALOG}.{schema_name} COMMENT 'Conformed domain: {schema_name} ({TARGET_ENV})'")
        print(f"  ✓ Silver Schema: {SILVER_CATALOG}.{schema_name}")
        
    # 4. Create Gold Consumption Schemas
    for schema_name in GOLD_SCHEMAS.values():
        spark.sql(f"CREATE SCHEMA IF NOT EXISTS {GOLD_CATALOG}.{schema_name} COMMENT 'Business consumption: {schema_name} ({TARGET_ENV})'")
        print(f"  ✓ Gold Schema: {GOLD_CATALOG}.{schema_name}")
        
    print(f"[INIT] Unity Catalog {TARGET_ENV.upper()} initialization complete.")

# Execute environment setup
init_environment()


In [0]:
# Common Audit & Hashing Utilities & Promotion Tools
from pyspark.sql.functions import current_timestamp, lit, sha2, concat_ws, coalesce, col

def add_audit_columns(df, run_id=None):
    """Appends standard ETL audit metadata columns."""
    out = df.withColumn("_ingest_timestamp", current_timestamp())
    if run_id:
        out = out.withColumn("_run_id", lit(run_id))
    return out

def generate_record_hash(df, tracked_columns):
    """Computes SHA-256 hash across tracked business attributes for change detection."""
    return df.withColumn(
        "record_hash",
        sha2(concat_ws("||", *[coalesce(col(c).cast("string"), lit("")) for c in tracked_columns]), 256)
    )

def clone_prod_to_dev(source_env="prod", target_env="dev"):
    """
    Zero-copy Delta SHALLOW CLONE utility to snapshot production tables into the dev catalogs
    for instant, realistic dev testing with zero additional storage cost.
    """
    print(f"\n[CLONE] Snapshotting {source_env.upper()} tables into {target_env.upper()} catalogs...")
    tables = [
        (f"bronze_{source_env}.ecommerce.orders", f"bronze_{target_env}.ecommerce.orders"),
        (f"bronze_{source_env}.ecommerce.products", f"bronze_{target_env}.ecommerce.products"),
        (f"bronze_{source_env}.ecommerce.customers", f"bronze_{target_env}.ecommerce.customers"),
        (f"bronze_{source_env}.ecommerce.region", f"bronze_{target_env}.ecommerce.region"),
        (f"silver_{source_env}.orders.orders", f"silver_{target_env}.orders.orders"),
        (f"silver_{source_env}.products.products", f"silver_{target_env}.products.products"),
        (f"silver_{source_env}.customers.customers", f"silver_{target_env}.customers.customers"),
        (f"silver_{source_env}.region.region", f"silver_{target_env}.region.region"),
    ]
    cloned_cnt = 0
    for src, tgt in tables:
        try:
            if spark.catalog.tableExists(src):
                spark.sql(f"CREATE OR REPLACE TABLE {tgt} SHALLOW CLONE {src}")
                print(f"  ✓ Cloned: {src} -> {tgt}")
                cloned_cnt += 1
            else:
                print(f"  [SKIP] Source table {src} does not exist.")
        except Exception as exc:
            print(f"  [WARN] Failed to clone {src} to {tgt}: {exc}")
    print(f"[CLONE] Complete: {cloned_cnt} tables cloned successfully to {target_env.upper()}.")

print("✓ Audit utilities, record_hash, and clone_prod_to_dev helper loaded.")
